# Monthly Table Egg Prices: Nominal and Real Analysis

This notebook analyzes monthly table egg prices from USDA NASS data. We load historical price data, adjust for inflation using CPI, and visualize both nominal and real price trends over time.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt  # figure creation and plot aesthetics
import seaborn as sns  # statistical plotting with built-in themes

sns.set_style('whitegrid')  # clean background with subtle gridlines

# Load egg price data from local CSV file
egg_prices = pd.read_csv('data/egg_table_prices_nass.csv')

# Load CPI data directly from GitHub URL
cpi = pd.read_csv('data/cpi_1913_2024.csv')

## Data Preparation

Filter the data to monthly observations and table eggs only, then create datetime variables for time series analysis.

In [ ]:
# Filter to monthly observations only (exclude annual summaries)
filter_years = ['YEAR', 'MARKETING YEAR']

egg_prices_monthly = egg_prices[~egg_prices['reference_period_desc'].isin(filter_years)].copy()

In [ ]:
# Filter to table eggs only (excludes hatching eggs and other classifications)
egg_prices_monthly = egg_prices_monthly[egg_prices_monthly['class_desc'].isin(['TABLE'])].copy()

In [ ]:
egg_prices_monthly.head()

### Create DateTime Variable

Convert year and month codes into a proper datetime format for time series plotting.

In [ ]:
# Create datetime column by combining year and begin_code (month)
# Set day=01 to represent the first day of each month
egg_prices_monthly['date'] = pd.to_datetime(
    egg_prices_monthly['year'].astype(str) + '-' + 
    egg_prices_monthly['begin_code'].astype(str) + '-01'
)

# Sort chronologically for proper time series plotting
egg_prices_monthly = egg_prices_monthly.sort_values('date').reset_index(drop=True)

# Verify datetime creation
print("First 5 observations:")
print(egg_prices_monthly[['year', 'reference_period_desc', 'begin_code', 'date', 'Value']].head())
print("\nLast 5 observations:")
print(egg_prices_monthly[['year', 'reference_period_desc', 'begin_code', 'date', 'Value']].tail())

In [ ]:
# Create complete monthly date range from first to last date in data
date_range = pd.date_range(
    start=egg_prices_monthly['date'].min(),
    end=egg_prices_monthly['date'].max(),
    freq='MS'  # Month Start frequency
)

# Find missing dates
actual_dates = set(egg_prices_monthly['date'])
expected_dates = set(date_range)
missing_dates = expected_dates - actual_dates

# Display results
if missing_dates:
    print(f"Missing {len(missing_dates)} month(s):")
    for date in sorted(missing_dates):
        print(f"  {date.strftime('%Y-%m')}")
else:
    print("No missing months - data is complete!")

# Summary
print(f"\nDate range: {egg_prices_monthly['date'].min().strftime('%Y-%m')} to {egg_prices_monthly['date'].max().strftime('%Y-%m')}")
print(f"Expected months: {len(expected_dates)}")
print(f"Actual months: {len(actual_dates)}")

## Inflation Adjustment

Merge with CPI data and calculate real prices adjusted to 2024 dollars.

In [ ]:
# Merge egg prices with annual CPI data
egg_prices_monthly = egg_prices_monthly.merge(
    cpi,
    left_on='year',
    right_on='year',
    how='left'
)

In [ ]:
# Calculate real prices in 2024 dollars
base_year_cpi = cpi[cpi['year'] == 2024]['annual_avg_cpi'].values[0]

egg_prices_monthly['real_price'] = egg_prices_monthly['Value'] * (base_year_cpi / egg_prices_monthly['annual_avg_cpi'])
egg_prices_monthly = egg_prices_monthly.rename(columns={'Value': 'nominal_price'})

In [ ]:
# Reorder columns to prioritize key variables
priority_cols = ['year', 'date', 'nominal_price', 'real_price']
new_order = priority_cols + [col for col in egg_prices_monthly.columns 
                              if col not in priority_cols]
egg_prices_monthly = egg_prices_monthly[new_order]

In [ ]:
# Filter to complete years only (exclude partial 2025 data)
egg_prices_monthly = egg_prices_monthly[egg_prices_monthly['year'] < 2025].copy()

## Visualization

Create time series plots showing both nominal and inflation-adjusted egg prices.

### Nominal Egg Prices

Plot monthly egg prices in current dollars.

In [ ]:
# Create figure with wider dimensions to accommodate monthly data points
fig, ax = plt.subplots(figsize=(14, 6))

sns.lineplot(
    data=egg_prices_monthly,
    x='date',
    y='nominal_price',
    linewidth=1.0,
    color='darkblue',
    errorbar=None,
    ax=ax
)

# Configure plot labels and styling
plt.title("Nominal Egg Prices (Monthly)", fontsize=14, weight='bold')
plt.xlabel("Year", fontsize=14, weight='bold')
plt.ylabel("Price ($/Dozen)", fontsize=14, weight='bold')

ax.tick_params(axis='x', rotation=0, labelsize=14)
ax.tick_params(axis='y', labelsize=14)

fig.tight_layout()
fig.savefig('output/figures/table_egg_prices_monthly_nominal.png', dpi=150, bbox_inches='tight')
plt.show()

### Real Egg Prices

Plot monthly egg prices adjusted to 2024 dollars using CPI.

In [ ]:
# Create figure with wider dimensions to accommodate monthly data points
fig, ax = plt.subplots(figsize=(14, 6))

sns.lineplot(
    data=egg_prices_monthly,
    x='date',
    y='real_price',
    linewidth=1.0,
    color='darkred',
    errorbar=None,
    ax=ax
)

# Configure plot labels and styling
plt.title("Real Egg Prices (Monthly, 2024 Dollars)", fontsize=14, weight='bold')
plt.xlabel("Year", fontsize=14, weight='bold')
plt.ylabel("Price ($/Dozen)", fontsize=14, weight='bold')

ax.tick_params(axis='x', rotation=0, labelsize=14)
ax.tick_params(axis='y', labelsize=14)

fig.tight_layout()
fig.savefig('output/figures/table_egg_prices_monthly_real.png', dpi=150, bbox_inches='tight')
plt.show()